# Multi-Round Feature Extraction

This notebook extends the original feature extraction to handle multiple multiplexing rounds.

**Key Features:**
- Shape features are extracted only from round 0 (where masks are located)
- Intensity features are extracted from all available rounds
- Cross-round correlations are calculated for the same stainings across rounds
- Automatic detection of available rounds in OME-Zarr data

# Imports

In [ ]:
# Import original functions
from Fcts_FE import make_experiment, estimate_staining_thresholds, threshold_change, plot_thresholds, test_skeletonization
from Fcts_Base import get_stainings, find_zarr_dirs
from Fcts_Plotting import segmentation_fidelity_check

# Import multi-round extensions
from Fcts_Base_MultiRound import extract_ome_zarr_tables_multiround, detect_available_rounds, get_available_rounds_for_barcode
from Fcts_FE_MultiRound import extract_features_multiround

%load_ext autoreload
%autoreload 2

# User Input

### General Settings

Provide information regarding directories, used barcodes, and define the file name for results.
Setup .xls file must be in source folder. See GitHub readme for more information.

In [ ]:
# Point to experiment folder. Has to include setup .xls file.
source = "YOUR PATH TO EXPERIMENT FOLDER"

# Optional: Analysis directory. If None this will default to the source folder.
analysis_dir = None

experiment_ID = "YOUR EXPERIMENT ID"

### Multi-Round Settings

Configure settings specific to multi-round multiplexing experiments.

In [ ]:
# Maximum number of rounds to process (None = auto-detect all available)
max_rounds = None  # e.g., 3 to process rounds 0, 1, 2

# Rounds to include in analysis (None = all detected rounds)
# This allows you to skip specific rounds if needed
rounds_to_include = None  # e.g., [0, 1, 3] to skip round 2

### Measurement Settings

In [ ]:
quantiles_to_calc = [0.01, 0.25, 0.50, 0.75, 0.99] # List of quantiles to be calculated

pyramid_level = 0 # Fractal pyramid level to use for feature extraction

table_name = "organoids_ROI_table" # Fractal ROI table name to load images

label_name = "organoids" # Fractal label name to load segmentation

# Load files and display experimental setup

Loads information from .xls setup files to subsequently link them to features of extracted organoids.

In [ ]:
stainings = get_stainings(source)

In [ ]:
folder = find_zarr_dirs(source) # Find ome_zarr files in source folder
experiment_setup, barcodes = make_experiment(source) # Create experiment setup from Layout

if analysis_dir is None:
    analysis_dir = source

# Load OME-Zarr plates with multi-round support

This enhanced loading function automatically detects and loads multiple multiplexing rounds.
**Important:** The mask is always extracted from round 0 (image_name="0").

In [ ]:
# Load all available rounds
ome_zarr_dict_multiround, ome_zarr_df = extract_ome_zarr_tables_multiround(
    experiment_setup, 
    source, 
    folder, 
    table_name,
    max_rounds=max_rounds
)

print(f"Successfully loaded {len(ome_zarr_dict_multiround)} plate-round combinations::")
for (barcode, round_num) in sorted(ome_zarr_dict_multiround.keys()):
    print(f"  - {barcode} Round {round_num}")

### Check available rounds per barcode

This cell shows which rounds are available for each barcode.

In [ ]:
print("Available rounds per barcode:")
for barcode in barcodes:
    available_rounds = get_available_rounds_for_barcode(ome_zarr_dict_multiround, barcode)
    print(f"  {barcode}: {available_rounds}")

# Estimate thresholds

**Note:** Thresholds are estimated from round 0 data and applied to all rounds.

### Automatic estimation

In [ ]:
control_condition = None # Must be key in Medium tab of .xls setup file
n = 10 # Number of organoids picked randomly per timepoint
seed = 50 # Random seed
sigma = 1 # Sigma for gaussian blurring
q = 0.5 # Quantile used to pick final threshold

# Use original function with round 0 data
ome_zarr_dict_round0 = {k[0]: v for k, v in ome_zarr_dict_multiround.items() if k[1] == 0}

thresholds, dict_org, timepoints_lst = estimate_staining_thresholds(
    ome_zarr_df=ome_zarr_df,
    ome_zarr_dict=ome_zarr_dict_round0, 
    stainings=stainings,
    experiment_setup=experiment_setup,
    table_name=table_name,
    label_name=label_name,
    pyramid_level=pyramid_level,
    control_condition=control_condition,
    n=n,
    seed=seed, 
    sigma=sigma,
    q=q
)

### Manual threshold adjustment (optional)

In [ ]:
# Example threshold change
# thresholds = threshold_change(
#     threshold=thresholds,
#     staining="DAPI",
#     new_threshold=1000
# )

### Test thresholds

In [ ]:
fig = plot_thresholds(
    thresholds=thresholds, 
    dict_org=dict_org, 
    timepoints_lst=timepoints_lst,
    ome_zarr_dict=ome_zarr_dict_round0,
    table_name=table_name,
    label_name=label_name,
    seed=0
)

# Visualize segmentation fidelity

Check segmentation quality using round 0 data.

In [ ]:
channel = [0] # Channel to plot
channel_colors = ["Gray"] # Color of channel to plot
channel_ranges = [[100,4000]] # Range of values to plot for channel
n = 1 # Number of wells to plot per timepoint
pyriamid_lvl_plot = 4 # OME_zarr Pyramid level to plot

segmentation_fidelity_check(
    ome_zarr_dict_round0, 
    channels=channel,
    channel_colors=channel_colors,
    channel_ranges=channel_ranges,
    n=n,
    label_name=label_name,
    scalebar_micrometer=100,
    pyramid_lvl_plot=pyriamid_lvl_plot
)

# Test skeleton features

Test skeleton parameters using round 0 data (where masks are located).

In [ ]:
sigma_skeleton = 5
n_angle_determination = 50
radius_multiplier = 0.8
n = 10
seed = 2

test_skeletonization(
    barcodes,
    ome_zarr_dict_round0,
    ome_zarr_df,
    table_name=table_name,
    label_name=label_name,
    pyramid_level=pyramid_level,
    n=n,
    seed=seed, 
    sigma_skeleton=sigma_skeleton, 
    n_angle_determination=n_angle_determination, 
    radius_multiplier=radius_multiplier
)

# Extract Features (Multi-Round)

This enhanced function extracts features from multiple multiplexing rounds:

- **Shape features** (area, perimeter, skeleton, etc.): Extracted only from round 0
- **Intensity features** (mean, std, quantiles, etc.): Extracted from all rounds
- **Cross-round correlations**: Calculated between the same stainings across rounds

Feature naming convention:
- Round 0 features: Normal names (e.g., "DAPI_mean")
- Other rounds: Round suffix (e.g., "DAPI_R1_mean", "DAPI_R2_mean")

In [ ]:
ad = extract_features_multiround(
    ome_zarrs_dict_multiround=ome_zarr_dict_multiround,
    table_name=table_name,
    label_name=label_name,
    pyramid_level=pyramid_level,
    source=source,
    folder=folder,
    analysis_dir=analysis_dir,
    barcodes=barcodes,
    experiment_setup=experiment_setup,
    thresholds=thresholds,
    stainings=stainings,
    result_file_name="1_FeatureExtraction",
    radius_multiplier=radius_multiplier,
    sigma_skeleton=sigma_skeleton,
    quantiles_to_calc=quantiles_to_calc,
    experiment_ID=experiment_ID,
    max_rounds=max_rounds
)

# Results Overview

Display summary of extracted features and available rounds.

In [ ]:
print(f"Successfully extracted features for {ad.n_obs} organoids")
print(f"Total features extracted: {ad.n_vars}")
print(f"Multiround analysis: {ad.uns.get('multiround', False)}")

# Show feature categories
feature_names = list(ad.var_names)
round_features = {}

for feat in feature_names:
    if '_R' in feat and feat.count('_R') == 1:
        # Extract round number
        parts = feat.split('_R')
        if len(parts) == 2 and parts[1].split('_')[0].isdigit():
            round_num = int(parts[1].split('_')[0])
            if round_num not in round_features:
                round_features[round_num] = 0
            round_features[round_num] += 1
    else:
        # Round 0 or shape features
        if 0 not in round_features:
            round_features[0] = 0
        round_features[0] += 1

print("\nFeatures per round:")
for round_num in sorted(round_features.keys()):
    print(f"  Round {round_num}: {round_features[round_num]} features")

# Show available rounds per organoid
print("\nAvailable rounds per organoid (first 10):")
for i in range(min(10, ad.n_obs)):
    oid = ad.obs.index[i]
    rounds = ad.obs.loc[oid, 'available_rounds']
    print(f"  {oid}: {rounds}")